# 08 — Evaluation: Fine-Tuned Model vs Gold Standard

Loads `Volcanex/baseline-schema-qwen3-vl-lora` (the trained LoRA adapter) on top of
Qwen3-VL-8B-Instruct and runs two evaluations:

1. **🎲 Random Visual Eval** — picks `N_EXAMPLES` random examples, shows screenshot +
   gold vs predicted JSON-LD side-by-side for a quick sanity check.

2. **📊 Batch Metrics** — runs on `N_BATCH` examples and reports JSON validity,
   @type accuracy, and property F1. Produces a 4-panel matplotlib report (charts saved
   inline and to `eval_report.png`).

3. **💾 Export** — saves the executed notebook as a self-contained HTML report.

**Run on RunPod (RTX 4090 / 3090 is fine — A100 is overkill for inference).**


In [ ]:
import os, sys, json, random, torch
import matplotlib
matplotlib.use('Agg')   # non-interactive backend — works in Jupyter AND nbconvert
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from pathlib import Path
from io import BytesIO
from collections import Counter
from dotenv import load_dotenv
from IPython.display import display, Image as IPImage, HTML

# ── Environment ───────────────────────────────────────────────────────────────
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

PROJECT_ROOT = Path('/workspace/schema')
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / '.env', override=True)

HF_TOKEN   = os.getenv('HF_TOKEN')
MODELS_DIR = Path('/workspace/models')

# Compatibility shim for bitsandbytes on some PyTorch builds
if not hasattr(torch.nn.Module, 'set_submodule'):
    def _set_submodule(self, target, module):
        parent, _, last = target.rpartition('.')
        parent_mod = self.get_submodule(parent) if parent else self
        setattr(parent_mod, last, module)
    torch.nn.Module.set_submodule = _set_submodule

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:  {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB')


## Load Model + LoRA Adapter


In [ ]:
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL  = 'Qwen/Qwen3-VL-8B-Instruct'
ADAPTER_HF  = 'Volcanex/baseline-schema-qwen3-vl-lora'
ADAPTER_SUB = 'final'   # use the end-of-epoch adapter

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
)

try:
    import flash_attn  # noqa
    attn_impl = 'flash_attention_2'
except ImportError:
    attn_impl = 'sdpa'

print(f'Loading base model ({BASE_MODEL})...')
model = Qwen3VLForConditionalGeneration.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    attn_implementation=attn_impl,
    device_map='auto',
    token=HF_TOKEN,
    cache_dir=str(MODELS_DIR),
)
model.eval()

print(f'Loading LoRA adapter ({ADAPTER_HF}/{ADAPTER_SUB})...')
model = PeftModel.from_pretrained(model, ADAPTER_HF, subfolder=ADAPTER_SUB, token=HF_TOKEN)
model.eval()

processor = AutoProcessor.from_pretrained(
    BASE_MODEL, token=HF_TOKEN, cache_dir=str(MODELS_DIR),
    min_pixels=256*28*28, max_pixels=640*28*28,
)

print(f'✓ Ready. VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB')


## Inference Helper


In [ ]:
from qwen_vl_utils import process_vision_info

def generate_schema(messages, max_new_tokens=600):
    """Run inference on one eval example. Returns predicted JSON-LD string."""
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    img_inputs, _ = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=img_inputs or None,
        return_tensors='pt',
    ).to(model.device)

    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,       # greedy — deterministic for eval
            temperature=None,
            top_p=None,
            use_cache=True,
        )

    # Strip the prompt tokens, decode only what the model generated
    new_tokens = out_ids[0][inputs['input_ids'].shape[1]:]
    return processor.tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

print('generate_schema() ready')


## 🎲 Random Eval — Visual Comparison


In [ ]:
from IPython.display import display, Image as IPImage, HTML
import base64

# ── Load eval set ─────────────────────────────────────────────────────────────
eval_path = PROJECT_ROOT / 'data' / 'processed' / 'eval.jsonl'
eval_examples = [json.loads(l) for l in open(eval_path)]
print(f'Eval examples: {len(eval_examples)}')

# ── Config ────────────────────────────────────────────────────────────────────
N_EXAMPLES = 5      # ← change this to see more/fewer examples
SEED       = 42     # ← change for different random picks

random.seed(SEED)
sample = random.sample(eval_examples, N_EXAMPLES)

# ── Display helper ────────────────────────────────────────────────────────────
def fmt_json(s):
    """Pretty-print a JSON string, or return raw if invalid."""
    try:
        return json.dumps(json.loads(s), indent=2, ensure_ascii=False)
    except Exception:
        return s

def show_comparison(ex, pred, idx):
    messages = ex['messages']
    gold = next(m['content'] for m in messages if m['role'] == 'assistant')
    img_path = None
    for m in messages:
        if m['role'] == 'user':
            for item in m.get('content', []):
                if isinstance(item, dict) and item.get('type') == 'image':
                    img_path = PROJECT_ROOT / item['image']

    # Parse types for quick headline
    try:
        pred_type = json.loads(pred).get('@type', '?')
    except Exception:
        pred_type = '⚠ invalid JSON'
    try:
        gold_type = json.loads(gold).get('@type', '?')
    except Exception:
        gold_type = '?'

    match_icon = '✅' if pred_type == gold_type else '❌'
    site_id = ex.get('id', f'example-{idx}')

    print(f'\n{"="*70}')
    print(f'  Example {idx+1}/{N_EXAMPLES} — {site_id}')
    print(f'  Gold @type: {gold_type}   Predicted @type: {pred_type}  {match_icon}')
    print(f'{"="*70}')

    # Screenshot
    if img_path and img_path.exists():
        display(IPImage(filename=str(img_path), width=500))
    else:
        print(f'  [screenshot not found: {img_path}]')

    # Side-by-side JSON comparison
    display(HTML(f"""
    <div style="display:grid;grid-template-columns:1fr 1fr;gap:16px;margin:8px 0;font-size:13px">
      <div>
        <b style="color:#2a7a2a">✅ Gold Standard</b>
        <pre style="background:#f0fff0;border:1px solid #ccc;padding:10px;
                    border-radius:4px;overflow:auto;max-height:400px">{fmt_json(gold)}</pre>
      </div>
      <div>
        <b style="color:#1a4fa8">🤖 Model Prediction</b>
        <pre style="background:#f0f4ff;border:1px solid #ccc;padding:10px;
                    border-radius:4px;overflow:auto;max-height:400px">{fmt_json(pred)}</pre>
      </div>
    </div>
    """))

# ── Run ───────────────────────────────────────────────────────────────────────
print(f'Running inference on {N_EXAMPLES} random examples (SEED={SEED})...\n')

results = []
for i, ex in enumerate(sample):
    print(f'  [{i+1}/{N_EXAMPLES}] generating...', end=' ', flush=True)
    pred = generate_schema(ex['messages'])
    print('done')
    results.append({'ex': ex, 'pred': pred})

print('\n── Results ──────────────────────────────────────────────')
for i, r in enumerate(results):
    show_comparison(r['ex'], r['pred'], i)


## 📊 Batch Metrics


In [ ]:
# ── Helpers ───────────────────────────────────────────────────────────────────
def extract_properties(jsonld_str):
    try:
        obj = json.loads(jsonld_str)
        return {k for k in obj if k not in {'@context','@type','@id','@graph'}}
    except Exception:
        return set()

def property_f1(pred_str, gold_str):
    pred = extract_properties(pred_str)
    gold = extract_properties(gold_str)
    if not gold:
        return 0.0
    tp = len(pred & gold)
    p  = tp / len(pred) if pred else 0.0
    r  = tp / len(gold)
    return 2*p*r/(p+r) if (p+r) else 0.0

def get_type(jsonld_str):
    try:
        t = json.loads(jsonld_str).get('@type', 'unknown')
        return t[0] if isinstance(t, list) else (t or 'unknown')
    except Exception:
        return 'parse_error'

def is_valid_json(s):
    try: json.loads(s); return True
    except Exception: return False

def type_match(pred_str, gold_str):
    try:
        pt = json.loads(pred_str).get('@type')
        gt = json.loads(gold_str).get('@type')
        ps = {pt} if isinstance(pt, str) else set(pt or [])
        gs = {gt} if isinstance(gt, str) else set(gt or [])
        return bool(ps & gs)
    except Exception:
        return False

# ── Config ────────────────────────────────────────────────────────────────────
N_BATCH = 50   # examples to run batch eval on (~10 min on RTX 3090)

print(f'Running batch eval on {N_BATCH} random examples...')
random.seed(99)
batch = random.sample(eval_examples, min(N_BATCH, len(eval_examples)))

batch_results = []
for i, ex in enumerate(batch):
    messages = ex['messages']
    gold = next(m['content'] for m in messages if m['role'] == 'assistant')
    pred = generate_schema(messages)
    batch_results.append({
        'id':         ex.get('id', f'ex-{i}'),
        'gold':       gold,
        'pred':       pred,
        'gold_type':  get_type(gold),
        'pred_type':  get_type(pred),
        'valid':      is_valid_json(pred),
        'type_match': type_match(pred, gold),
        'f1':         property_f1(pred, gold),
    })
    flag = '✅' if batch_results[-1]['type_match'] else '❌'
    print(f'  [{i+1:02d}/{len(batch)}] {flag}  f1={batch_results[-1]["f1"]:.2f}  '
          f'{batch_results[-1]["gold_type"]} → {batch_results[-1]["pred_type"]}')

# ── Summary table ─────────────────────────────────────────────────────────────
validity_rate  = np.mean([r['valid'] for r in batch_results])
type_accuracy  = np.mean([r['type_match'] for r in batch_results])
avg_f1         = np.mean([r['f1'] for r in batch_results])

print(f'\n{"─"*42}')
print(f'  Eval set size    : {len(eval_examples)}')
print(f'  Batch sampled    : {len(batch_results)}')
print(f'  JSON validity    : {validity_rate*100:.1f}%')
print(f'  @type accuracy   : {type_accuracy*100:.1f}%')
print(f'  Property F1 avg  : {avg_f1:.3f}')
print(f'{"─"*42}')


## 📈 Charts


In [ ]:
PALETTE = {
    'green':  '#27ae60',
    'blue':   '#2980b9',
    'purple': '#8e44ad',
    'orange': '#e67e22',
    'red':    '#e74c3c',
    'grey':   '#95a5a6',
}

fig, axes = plt.subplots(2, 2, figsize=(15, 11))
fig.suptitle(
    f'Baseline Schema Model — Eval Report  |  n={len(batch_results)}  |  '
    f'Validity {validity_rate*100:.0f}%  @type acc {type_accuracy*100:.0f}%  F1 {avg_f1:.2f}',
    fontsize=13, fontweight='bold', y=1.01,
)

# ── 1. Overall metrics ────────────────────────────────────────────────────────
ax = axes[0, 0]
metrics  = ['JSON\nValidity', '@type\nAccuracy', 'Property\nF1']
values   = [validity_rate, type_accuracy, avg_f1]
colors   = [PALETTE['green'], PALETTE['blue'], PALETTE['purple']]
bars = ax.bar(metrics, [v * 100 for v in values], color=colors, width=0.5, edgecolor='white')
ax.set_ylim(0, 110)
ax.set_ylabel('Score (%)')
ax.set_title('Overall Metrics', fontweight='bold')
ax.axhline(100, color='#ccc', linestyle='--', linewidth=0.8)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2,
            f'{val*100:.1f}%', ha='center', fontsize=11, fontweight='bold')
ax.spines[['top', 'right']].set_visible(False)

# ── 2. Property F1 histogram ──────────────────────────────────────────────────
ax = axes[0, 1]
f1_vals = [r['f1'] for r in batch_results]
ax.hist(f1_vals, bins=20, range=(0, 1), color=PALETTE['blue'], edgecolor='white', alpha=0.85)
mean_f1 = np.mean(f1_vals)
med_f1  = np.median(f1_vals)
ax.axvline(mean_f1, color=PALETTE['red'],    linestyle='--', linewidth=1.5,
           label=f'Mean={mean_f1:.2f}')
ax.axvline(med_f1,  color=PALETTE['orange'], linestyle=':',  linewidth=1.5,
           label=f'Median={med_f1:.2f}')
ax.set_xlabel('Property F1 Score')
ax.set_ylabel('Count')
ax.set_title('Property F1 Distribution', fontweight='bold')
ax.legend(fontsize=9)
ax.spines[['top', 'right']].set_visible(False)

# ── 3. @type distribution (gold vs pred) ─────────────────────────────────────
ax = axes[1, 0]
gold_ctr  = Counter(r['gold_type'] for r in batch_results)
pred_ctr  = Counter(r['pred_type'] for r in batch_results)
top_types = [t for t, _ in gold_ctr.most_common(8)]
x = np.arange(len(top_types))
w = 0.35
ax.bar(x - w/2, [gold_ctr[t] for t in top_types], w,
       label='Gold', color=PALETTE['green'], edgecolor='white')
ax.bar(x + w/2, [pred_ctr.get(t, 0) for t in top_types], w,
       label='Predicted', color=PALETTE['blue'], edgecolor='white', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(top_types, rotation=35, ha='right', fontsize=9)
ax.set_ylabel('Count')
ax.set_title('@type Distribution  (top 8)', fontweight='bold')
ax.legend(fontsize=9)
ax.spines[['top', 'right']].set_visible(False)

# ── 4. Per-@type accuracy ─────────────────────────────────────────────────────
ax = axes[1, 1]
type_groups = {}
for r in batch_results:
    type_groups.setdefault(r['gold_type'], []).append(r['type_match'])
# keep types with at least 2 examples, sorted by accuracy descending
per_type = {t: np.mean(v) for t, v in type_groups.items() if len(v) >= 2}
sorted_items = sorted(per_type.items(), key=lambda x: -x[1])[:10]
labels = [t for t, _ in sorted_items]
vals   = [v * 100 for _, v in sorted_items]
bar_colors = [PALETTE['green'] if v == 100 else PALETTE['blue'] if v >= 50 else PALETTE['red']
              for v in vals]
ax.barh(labels, vals, color=bar_colors, edgecolor='white')
ax.set_xlabel('@type Accuracy (%)')
ax.set_title('Per-@type Accuracy  (min 2 examples)', fontweight='bold')
ax.set_xlim(0, 115)
for i, v in enumerate(vals):
    ax.text(v + 1, i, f'{v:.0f}%', va='center', fontsize=9)
ax.invert_yaxis()
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()

# Save to file AND embed inline
report_png = PROJECT_ROOT / 'notebooks' / 'eval_report.png'
fig.savefig(str(report_png), dpi=150, bbox_inches='tight')

buf = BytesIO()
fig.savefig(buf, format='png', dpi=130, bbox_inches='tight')
buf.seek(0)
display(IPImage(data=buf.getvalue()))
plt.close(fig)

print(f'\n✓ Chart saved to {report_png}')


## 💾 Export Report


In [ ]:
import subprocess, shutil

NB_PATH   = PROJECT_ROOT / 'notebooks' / '08_evaluation.ipynb'
HTML_OUT  = PROJECT_ROOT / 'notebooks' / 'eval_report'   # nbconvert adds .html

print('Converting executed notebook to HTML...')
result = subprocess.run(
    ['jupyter', 'nbconvert', '--to', 'html', '--no-input',
     str(NB_PATH), '--output', str(HTML_OUT)],
    capture_output=True, text=True,
)
if result.returncode != 0:
    print('nbconvert stderr:', result.stderr[-500:])
else:
    html_file = Path(str(HTML_OUT) + '.html')
    size_mb   = html_file.stat().st_size / 1e6
    print(f'✓ Report saved: {html_file}  ({size_mb:.1f} MB)')
    print()
    print('── To copy the report to your local machine ──────────────────')
    print('Run this on your LOCAL machine (fill in SSH details):')
    print()
    # Try to auto-detect pod SSH port from env
    pod_ip   = os.getenv('RUNPOD_POD_HOSTNAME', '<pod-ip>')
    pod_port = os.getenv('RUNPOD_SSH_PORT', '<port>')
    print(f'  scp -P {pod_port} root@{pod_ip}:{html_file} ~/schema/notebooks/eval_report.html')
    print()
    print('Or download directly from JupyterLab: File browser → notebooks/ → eval_report.html → right-click → Download')
